In [0]:
import pandas as pd
from pyspark.sql import functions as F

schema = 'finance'
table_name = 'dim_taxonomy'

dbutils.widgets.text("year", "", "GAAP Version Year (leave blank for all versions)")
gaap_year_to_process = dbutils.widgets.get("year")

dbutils.widgets.text("target_catalog", "", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")

# -----------------------------------------------------------------
# 1. Load staging data — optionally scoped to a single GAAP version
# -----------------------------------------------------------------
staging = spark.table("operations.finance_staging.dim_taxonomy_staging")

if gaap_year_to_process:
    gaap_version = f"us-gaap/{gaap_year_to_process}"
    staging = staging.filter(F.col("gaap_version") == gaap_version)

df = staging.toPandas()

# -----------------------------------------------------------------
# 2. Load fact leaf nodes — version-aware so each (leaf, version)
#    pair is treated independently
# -----------------------------------------------------------------
fact = (
    spark.table("operations.finance_staging.fact_staging_financial_statement")
    .select("terse_label", "gaap_version")
    .distinct()
    .toPandas()
)

# Set of (terse_label, gaap_version) pairs that actually exist in fact
fact_pairs = set(zip(fact["terse_label"], fact["gaap_version"]))

# -----------------------------------------------------------------
# 3. Build parent map — keyed on (child_label, gaap_version, linkrole)
# -----------------------------------------------------------------
parent_map = {
    (row["child_label"], row["gaap_version"], row["linkrole"]): row["parent_label"]
    for _, row in df.iterrows()
}

# -----------------------------------------------------------------
# 4. Map each child label → all (gaap_version, linkrole) combos it
#    appears in across the full staging table
# -----------------------------------------------------------------
child_version_linkrole_map = (
    df.groupby("child_label")
    .apply(lambda x: list(zip(x["gaap_version"], x["linkrole"])))
    .to_dict()
)

# -----------------------------------------------------------------
# 5. Path builder — unchanged; walks parent_map up to max_depth
# -----------------------------------------------------------------
def build_path(child, gaap_version, linkrole, parent_map, max_depth=50):
    path = []
    current = child
    visited = set()
    for _ in range(max_depth):
        key = (current, gaap_version, linkrole)
        if current is None or key in visited:
            break
        path.append(current)
        visited.add(key)
        current = parent_map.get(key)
    return path[::-1]

# -----------------------------------------------------------------
# 6. Build paths — only for (leaf, version) pairs present in fact,
#    spanning all linkroles that leaf appears in for that version
# -----------------------------------------------------------------
paths = []
for leaf, version in fact_pairs:
    version_linkrole_pairs = [
        (v, lr) for v, lr in child_version_linkrole_map.get(leaf, [])
        if v == version
    ]
    for _, linkrole in version_linkrole_pairs:
        path = build_path(leaf, version, linkrole, parent_map)
        paths.append({
            "leaf_node": leaf,
            "gaap_version": version,
            "linkrole": linkrole,
            "path": path
        })

paths_df = pd.DataFrame(paths)
max_depth = paths_df["path"].apply(len).max()

# -----------------------------------------------------------------
# 7. Expand path list into level columns
#    NOTE: label_map (identity mapping) removed — it was a no-op
# -----------------------------------------------------------------
for i in range(max_depth):
    paths_df[f"level_{i}"] = paths_df["path"].apply(
        lambda x, i=i: x[i] if i < len(x) else None
    )

spark.createDataFrame(paths_df).createOrReplaceTempView('df')

# -----------------------------------------------------------------
# 8. Final select — surrogate keys on leaf, version, and linkrole
# -----------------------------------------------------------------
final_df = spark.sql(f"""
select
     bigint(substr(xxhash64(concat_ws('|', leaf_node)), 1, 18))        AS terse_label_bigint_key
    ,sha2(concat_ws('|', leaf_node), 256)                               AS terse_label_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', gaap_version)), 1, 18))      AS gaap_version_bigint_key
    ,sha2(concat_ws('|', gaap_version), 256)                            AS gaap_version_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', linkrole)), 1, 18))          AS linkrole_bigint_key
    ,sha2(concat_ws('|', linkrole), 256)                                AS linkrole_key_hash
    ,gaap_version
    ,linkrole
    ,leaf_node                                                           AS terse_label
    ,level_1                                                             AS terse_label_level_1
    ,level_2                                                             AS terse_label_level_2
    ,level_3                                                             AS terse_label_level_3
    ,level_4                                                             AS terse_label_level_4
    ,level_5                                                             AS terse_label_level_5
    ,level_6                                                             AS terse_label_level_6
    ,level_7                                                             AS terse_label_level_7
    ,level_8                                                             AS terse_label_level_8
    ,level_9                                                             AS terse_label_level_9
    ,level_10                                                            AS terse_label_level_10
    ,level_11                                                            AS terse_label_level_11
from df
""").createOrReplaceTempView('df')

#final_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target_catalog}.{schema}.{table_name}")